# Sales Agent Research

## Overview
This notebook demonstrates how to build a PII protection system for a sales agent chatbot.

The core problem: When a user shares sensitive information like name, email, or phone number in a chatbot, that data should never be exposed to the LLM, stored raw in Redis, or leaked into conversation history.

## What Was Built
- PII detection using Microsoft Presidio
- Tokenization system that replaces real values with tokens
- Dual query architecture for RAG and LLM
- Tool call interception so tools get real values without LLM knowing
- Redis storage with encrypted token map and metadata
- LangGraph agent connecting everything together

## Problem Statement

In a live chatbot, users share sensitive information:
- Name
- Email
- Phone number
- Account details

Without protection this happens:
User: "My email is david@gmail.com"
LLM API receives raw email
Sits in OpenAI servers
Appears in logs
Stored in Redis history
User asks again -> LLM recalls real email

This violates GDPR, breaks compliance, and exposes customer data.

### Our Solution
Two layer architecture:

* Layer 1 - Deterministic: Fast rule based checks, regex, hard blocks
* Layer 2 - Hybrid: Presidio NER + tokenization + dual query

Both layers run before any user input reaches the LLM.

### Detect phone number
Added Google's phonenumbers library to replace Presidio's weak US-centric phone regex with a solution that covers 240+ country formats.
Built a custom StrictPhoneRecognizer that loops through phonenumbers.SUPPORTED_REGIONS and registered it via analyzer.registry.

 ## Install all Libraries

In [ ]:
!pip install presidio-analyzer presidio-anonymizer
!pip install openai langgraph
!pip install redis fakeredis cryptography
!pip install phonenumbers
!python -m spacy download en_core_web_lg

print("All libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.9/499.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All libraries installed


In [ ]:
!pip install phonenumbers

## Setup All Engines

Setting up Presidio engines and OpenAI client.
AnalyzerEngine detects PII.
AnonymizerEngine available but we use manual tokenization for full control.

In [ ]:
import os
from functools import lru_cache

import phonenumbers
from openai import OpenAI
from presidio_analyzer import AnalyzerEngine, EntityRecognizer, RecognizerResult
from presidio_anonymizer import AnonymizerEngine

# Presidio engines
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()


@lru_cache(maxsize=512)
def detect_phones(text):
    seen_spans = set()
    found = []
    for region in phonenumbers.SUPPORTED_REGIONS:  # all 240+ countries
        for match in phonenumbers.PhoneNumberMatcher(text, region):
            span = (match.start, match.end)
            if span not in seen_spans:
                seen_spans.add(span)
                found.append((match.start, match.end))
    for match in phonenumbers.PhoneNumberMatcher(text, None):  # +country_code formats
        span = (match.start, match.end)
        if span not in seen_spans:
            seen_spans.add(span)
            found.append((match.start, match.end))
    return found


class StrictPhoneRecognizer(EntityRecognizer):
    def load(self):
        pass

    def analyze(self, text, entities, nlp_artifacts=None):
        return [
            RecognizerResult(entity_type="PHONE_NUMBER", start=s, end=e, score=0.95)
            for s, e in detect_phones(text)
        ]


analyzer.registry.remove_recognizer("PhoneRecognizer")  # remove weak built-in
analyzer.registry.add_recognizer(
    StrictPhoneRecognizer(supported_entities=["PHONE_NUMBER"], supported_language="en")
)

# OpenAI client
os.environ["OPENAI_API_KEY"] = "[API KEY]"
client = OpenAI()

print("Presidio Analyzer ready")
print("Presidio Anonymizer ready")
print("OpenAI client ready")

Presidio Analyzer ready
Presidio Anonymizer ready
OpenAI client ready


## Basic PII Detection

In [ ]:
text = "Hi, my name is David Sharma. My email is david@gmail.com and my phone is 9841234567."

results = analyzer.analyze(text=text, language="en")

print("Detected Entities:\n")
for r in results:
    print(f"  Type       : {r.entity_type}")
    print(f"  Value      : '{text[r.start : r.end]}'")
    print(f"  Confidence : {r.score:.2f}")
    print()

Detected Entities:

  Type       : EMAIL_ADDRESS
  Value      : 'david@gmail.com'
  Confidence : 1.00

  Type       : PHONE_NUMBER
  Value      : '9841234567'
  Confidence : 0.95

  Type       : PERSON
  Value      : 'David Sharma'
  Confidence : 0.85

  Type       : URL
  Value      : 'gmail.com'
  Confidence : 0.50

  Type       : US_BANK_NUMBER
  Value      : '9841234567'
  Confidence : 0.05

  Type       : US_DRIVER_LICENSE
  Value      : '9841234567'
  Confidence : 0.01



## Tokenization System (Token Map)

In [ ]:
token_map = {}
entity_counters = {}

EXCLUDE_ENTITIES = ["URL", "US_BANK_NUMBER", "US_DRIVER_LICENSE", "DATE_TIME"]


def remove_overlaps(results):
    # If two detections overlap, keep the higher confidence one
    results = sorted(results, key=lambda x: x.score, reverse=True)
    final = []
    for r in results:
        overlapping = False
        for f in final:
            if r.start < f.end and r.end > f.start:
                overlapping = True
                break
        if not overlapping:
            final.append(r)
    return final


def tokenize_text(text, threshold=0.4):
    # Scan the text to find all sensitive data
    results = analyzer.analyze(text=text, language="en")

    # Filter by confidence threshold and exclude noisy entities
    results = [
        r
        for r in results
        if r.score >= threshold and r.entity_type not in EXCLUDE_ENTITIES
    ]

    # Sort right to left so replacing does not shift positions
    results = sorted(results, key=lambda x: x.start, reverse=True)

    sanitized = text
    for entity in results:
        original_value = text[entity.start : entity.end]
        entity_type = entity.entity_type

        # Check if same value already has a token
        existing_token = None
        for token, value in token_map.items():
            if value == original_value:
                existing_token = token
                break

        if existing_token:
            token = existing_token
        else:
            if entity_type not in entity_counters:
                entity_counters[entity_type] = 1
            else:
                entity_counters[entity_type] += 1

            token = f"<{entity_type}_{entity_counters[entity_type]}>"
            token_map[token] = original_value

        sanitized = sanitized[: entity.start] + token + sanitized[entity.end :]

    return sanitized


token_map.clear()
entity_counters.clear()

text = "Hi, my name is David Sharma. My email is david@gmail.com and my phone is 9841234567."
sanitized = tokenize_text(text)

print(f"Original  : {text}")
print(f"Sanitized : {sanitized}")
print("\nToken Map :")
for token, value in token_map.items():
    print(f"  {token} -> {value}")

Original  : Hi, my name is David Sharma. My email is david@gmail.com and my phone is 9841234567.
Sanitized : Hi, my name is <PERSON_1>. My email is <EMAIL_ADDRESS_1> and my phone is <PHONE_NUMBER_1>.

Token Map :
  <PHONE_NUMBER_1> -> 9841234567
  <EMAIL_ADDRESS_1> -> david@gmail.com
  <PERSON_1> -> David Sharma


### Credit Card block
User message comes in
    -> Check for hard block entities
    -> If credit card found -> reject immediately, pipeline stops
    -> If clean -> pass to tokenization

In [ ]:
HARD_BLOCK_ENTITIES = ["CREDIT_CARD"]


def input_guardrail(text):

    results = analyzer.analyze(
        text=text,
        language="en",
        entities=["CREDIT_CARD", "PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"],
    )

    for entity in results:
        if entity.entity_type in HARD_BLOCK_ENTITIES:
            return {
                "blocked": True,
                "reason": entity.entity_type,
                "message": f"For your security, please do not share {entity.entity_type.replace('_', ' ').title()} details in the chat.",
            }

    return {"blocked": False}


# Test 1 - normal message
result = input_guardrail("What are your loan rates?")
print("Test 1:")
print(f"  Blocked : {result['blocked']}")

# Test 2 - credit card present
result = input_guardrail("My card number is 4532015112830366, book me a meeting")
print("\nTest 2:")
print(f"  Blocked : {result['blocked']}")
if result["blocked"]:
    print(f"  Reason  : {result['reason']}")
    print(f"  Message : {result['message']}")

Test 1:
  Blocked : False

Test 2:
  Blocked : True
  Reason  : CREDIT_CARD
  Message : For your security, please do not share Credit Card details in the chat.


## Dual Query (rag_query vs llm_query)

In [ ]:
def create_dual_queries(text):

    # Step 1 - tokenize everything, this becomes llm_query
    llm_query = tokenize_text(text)

    # Step 2 - for rag_query restore only PERSON tokens
    # EMAIL, PHONE, AMOUNT stay as tokens
    rag_query = llm_query
    for token, value in token_map.items():
        if "PERSON" in token:
            rag_query = rag_query.replace(token, value)

    return rag_query, llm_query


# test name ,uses NER
text = "My name is Ram Sharma, what are the FD rates?"
rag_query, llm_query = create_dual_queries(text)

print("Test 1:")
print(f"  Original  : {text}")
print(f"  rag_query : {rag_query}")
print(f"  llm_query : {llm_query}")
print(f"  Token Map : {token_map}")

# test name + email + amount
text = "My name is David Sharma, email david@gmail.com, budget 80000"
rag_query, llm_query = create_dual_queries(text)

print("\nTest 2:")
print(f"  Original  : {text}")
print(f"  rag_query : {rag_query}")
print(f"  llm_query : {llm_query}")
print(f"  Token Map : {token_map}")

Test 1:
  Original  : My name is Ram Sharma, what are the FD rates?
  rag_query : My name is Ram Sharma, what are the FD rates?
  llm_query : My name is <PERSON_2>, what are the FD rates?
  Token Map : {'<PHONE_NUMBER_1>': '9841234567', '<EMAIL_ADDRESS_1>': 'david@gmail.com', '<PERSON_1>': 'David Sharma', '<PERSON_2>': 'Ram Sharma'}

Test 2:
  Original  : My name is David Sharma, email david@gmail.com, budget 80000
  rag_query : My name is David Sharma, email <EMAIL_ADDRESS_1>, budget <PHONE_NUMBER_2>
  llm_query : My name is <PERSON_1>, email <EMAIL_ADDRESS_1>, budget <PHONE_NUMBER_2>
  Token Map : {'<PHONE_NUMBER_1>': '9841234567', '<EMAIL_ADDRESS_1>': 'david@gmail.com', '<PERSON_1>': 'David Sharma', '<PERSON_2>': 'Ram Sharma', '<PHONE_NUMBER_2>': '80000'}


## Multi-Turn Conversation

In [ ]:
token_map.clear()
entity_counters.clear()
conversation_history = []


def process_turn(user_message):
    print(f"User Original : {user_message}")

    # Step 1 - hard block check
    guardrail = input_guardrail(user_message)
    if guardrail["blocked"]:
        print(f"System        : {guardrail['message']}")
        print()
        return

    # Step 2 - create dual queries
    _rag_query, llm_query = create_dual_queries(user_message)

    # Step 3 - save llm_query to history, never original
    conversation_history.append({"role": "user", "content": llm_query})

    print(f"LLM sees      : {llm_query}")
    print(f"Token Map     : {token_map}")
    print()


# booking conversation
process_turn("Hi my name is David Sharma")
process_turn("My email is david@gmail.com")
process_turn("Please confirm booking to david@gmail.com")
process_turn("My card is 4532015112830366")

print("Conversation History (what LLM sees across all turns):")
for i, turn in enumerate(conversation_history):
    print(f"  Turn {i + 1} : {turn['content']}")

User Original : Hi my name is David Sharma
LLM sees      : Hi my name is <PERSON_1>
Token Map     : {'<PERSON_1>': 'David Sharma'}

User Original : My email is david@gmail.com
LLM sees      : My email is <EMAIL_ADDRESS_1>
Token Map     : {'<PERSON_1>': 'David Sharma', '<EMAIL_ADDRESS_1>': 'david@gmail.com'}

User Original : Please confirm booking to david@gmail.com
LLM sees      : Please confirm booking to <EMAIL_ADDRESS_1>
Token Map     : {'<PERSON_1>': 'David Sharma', '<EMAIL_ADDRESS_1>': 'david@gmail.com'}

User Original : My card is 4532015112830366
System        : For your security, please do not share Credit Card details in the chat.

Conversation History (what LLM sees across all turns):
  Turn 1 : Hi my name is <PERSON_1>
  Turn 2 : My email is <EMAIL_ADDRESS_1>
  Turn 3 : Please confirm booking to <EMAIL_ADDRESS_1>


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "book_meeting",
            "description": "Books a meeting for the customer when you have their name, email and phone",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Full name of the customer",
                    },
                    "email": {
                        "type": "string",
                        "description": "Email address of the customer",
                    },
                    "phone": {
                        "type": "string",
                        "description": "Phone number of the customer",
                    },
                },
                "required": ["name", "email", "phone"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_loan_rates",
            "description": "Fetches current loan or FD rates for the customer",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_type": {
                        "type": "string",
                        "description": "Type of product, for example FD, loan, savings",
                    }
                },
                "required": ["product_type"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_crm_lead",
            "description": "Creates a new lead in CRM system when customer shows interest",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Full name of the customer",
                    },
                    "email": {
                        "type": "string",
                        "description": "Email address of the customer",
                    },
                    "phone": {
                        "type": "string",
                        "description": "Phone number of the customer",
                    },
                    "interest": {
                        "type": "string",
                        "description": "What product the customer is interested in",
                    },
                },
                "required": ["name", "phone", "interest"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "send_confirmation_email",
            "description": "Sends a confirmation email to the customer after booking",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {
                        "type": "string",
                        "description": "Email address to send confirmation to",
                    },
                    "message": {
                        "type": "string",
                        "description": "Confirmation message to send",
                    },
                },
                "required": ["email", "message"],
            },
        },
    },
]

print(f"Defined {len(tools)} tools:")
for t in tools:
    print(f"  - {t['function']['name']}: {t['function']['description']}")

Defined 4 tools:
  - book_meeting: Books a meeting for the customer when you have their name, email and phone
  - check_loan_rates: Fetches current loan or FD rates for the customer
  - create_crm_lead: Creates a new lead in CRM system when customer shows interest
  - send_confirmation_email: Sends a confirmation email to the customer after booking


## Tool Call Interception


In [ ]:
import json


def restore_tokens(data):
    # Restore real values from token map
    # Works on both strings and dicts
    # Called only at tool execution, never by LLM
    if isinstance(data, str):
        result = data
        for token, real_value in token_map.items():
            result = result.replace(token, real_value)
        return result
    elif isinstance(data, dict):
        return {k: restore_tokens(v) for k, v in data.items()}


def run_agent():
    messages = [
        {
            "role": "system",
            "content": """You are a helpful sales agent for a bank.
When user wants to book a meeting and you have their name, email and phone, call the book_meeting tool.
For checking rates you just need the product type.
For creating a lead you need name, phone and their interest.
If you are missing information ask for it politely.""",
        }
    ]

    messages.extend(conversation_history)

    print(f"\n  LLM prompt includes (user part): '{messages[-1]['content']}'")

    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, tools=tools, tool_choice="auto"
    )

    response_message = response.choices[0].message

    tokenized_agent_response_for_history = ""
    user_facing_agent_response = ""

    if response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            tool_name = tool_call.function.name
            llm_params = json.loads(tool_call.function.arguments)

            print("\n  LLM decided to call a tool:")
            print(f"    Tool name        : {tool_name}")
            print(f"    LLM sends tokens : {llm_params}")

            # Intercept and restore real values
            real_params = restore_tokens(llm_params)
            print(f"    System restores  : {real_params}")

            # Tool executes with real values
            tool_result = {
                "status": "success",
                "message": f"Meeting booked for {real_params.get('name', '')}, confirmation to {real_params.get('email', '')}, will call {real_params.get('phone', '')}",
            }

            # Immediately sanitize tool result (Output Guardrail)
            sanitized = output_guardrail(tool_result["message"])
            print(f"  Tool result (real)      : {tool_result['message']}")
            print(f"  Tool result (sanitized) : {sanitized}")

            # Restore for user display
            final = restore_for_user(sanitized)
            print(f"\nUser sees (from tool) : {final}")

            tokenized_agent_response_for_history = sanitized
            user_facing_agent_response = final
    else:
        raw = response_message.content
        print(f"\n  LLM raw response : {raw}")

        # Apply Output Guardrail
        sanitized = output_guardrail(raw)
        print(f"  LLM response (sanitized by guardrail): {sanitized}")

        final = restore_for_user(sanitized)
        print(f"User sees (from LLM)    : {final}")

        tokenized_agent_response_for_history = sanitized
        user_facing_agent_response = final

    return tokenized_agent_response_for_history, user_facing_agent_response


print("run_agent function modified for proper return and global history usage.")

run_agent function modified for proper return and global history usage.


## Output Gradrails

In [ ]:
def output_guardrail(llm_response):
    results = analyzer.analyze(text=llm_response, language="en")

    # Filter same way as input
    results = [
        r for r in results if r.score >= 0.4 and r.entity_type not in EXCLUDE_ENTITIES
    ]

    if not results:
        return llm_response

    # Clean it by tokenizing again
    cleaned = tokenize_text(llm_response)
    return cleaned


def restore_for_user(text):
    # Restore tokens back to real values for user display
    result = text
    for token, real_value in token_map.items():
        result = result.replace(token, real_value)
    return result

## Redis Setup

In [ ]:
import fakeredis
from cryptography.fernet import Fernet

r = fakeredis.FakeRedis()

ENCRYPTION_KEY = Fernet.generate_key()
fernet = Fernet(ENCRYPTION_KEY)

print("Redis connected")
print("Encryption ready")

Redis connected
Encryption ready


## Redis Save Functions


In [ ]:
def save_to_redis(session_id, conversation_history, token_map, metadata):

    # Save 1 - History with tokens only
    history_key = f"{session_id}:history"
    r.set(history_key, json.dumps(conversation_history))
    print(f"  History saved    -> {history_key}")
    print("  Content          -> tokens only, no real PII")

    # Save 2 - Token map encrypted
    # Real values stored her
    tokenmap_key = f"{session_id}:token_map"
    tokenmap_raw = json.dumps(token_map).encode()
    tokenmap_encrypted = fernet.encrypt(tokenmap_raw)
    r.set(tokenmap_key, tokenmap_encrypted)
    print(f"  Token map saved  -> {tokenmap_key} (encrypted)")
    print("  Content          -> real values, encrypted")

    # Save 3 - Metadata encrypted
    metadata_key = f"{session_id}:metadata"
    metadata_raw = json.dumps(metadata).encode()
    metadata_encrypted = fernet.encrypt(metadata_raw)
    r.set(metadata_key, metadata_encrypted)
    print(f"  Metadata saved   -> {metadata_key} (encrypted)")
    print("  Content          -> structured contact info for dashboard")


def get_history_from_redis(session_id):
    key = f"{session_id}:history"
    data = r.get(key)
    if data:
        return json.loads(data)
    return []


def get_tokenmap_from_redis(session_id):
    key = f"{session_id}:token_map"
    encrypted = r.get(key)
    if encrypted:
        decrypted = fernet.decrypt(encrypted)
        return json.loads(decrypted)
    return {}


def get_metadata_from_redis(session_id):
    key = f"{session_id}:metadata"
    encrypted = r.get(key)
    if encrypted:
        decrypted = fernet.decrypt(encrypted)
        return json.loads(decrypted)
    return {}


print("Redis save and read functions ready")

Redis save and read functions ready


## Build Metadata From Token Map

In [ ]:
def build_metadata_from_tokenmap(token_map):

    metadata = {"fullname": None, "email": None, "phonenumber": None}

    for token, real_value in token_map.items():
        if "NAME" in token or "PERSON" in token:
            metadata["fullname"] = real_value
        elif "EMAIL" in token:
            metadata["email"] = real_value
        elif "PHONE" in token:
            metadata["phonenumber"] = real_value

    return metadata


token_map.clear()
entity_counters.clear()

text = "My name is Ram Sharma, email ram@gmail.com, phone 9841234567"
tokenize_text(text)

metadata = build_metadata_from_tokenmap(token_map)

print("Token Map:")
for token, value in token_map.items():
    print(f"  {token} -> {value}")

print("\nMetadata built from token_map:")
print(f"  fullname    : {metadata['fullname']}")
print(f"  email       : {metadata['email']}")
print(f"  phonenumber : {metadata['phonenumber']}")

Token Map:
  <PHONE_NUMBER_1> -> 9841234567
  <EMAIL_ADDRESS_1> -> ram@gmail.com
  <PERSON_1> -> Ram Sharma

Metadata built from token_map:
  fullname    : Ram Sharma
  email       : ram@gmail.com
  phonenumber : 9841234567


## Dashboard View

In [ ]:
session_id = "session_bank_001"


def dashboard_view(session_id):
    print("\n" + "=" * 55)
    print("BANK STAFF DASHBOARD")
    print("=" * 55)

    # Read metadata from Redis
    # This is what bank staff sees
    metadata = get_metadata_from_redis(session_id)

    print("\nCustomer Contact Details:")
    print(f"  Name  : {metadata.get('fullname', 'Not provided')}")
    print(f"  Email : {metadata.get('email', 'Not provided')}")
    print(f"  Phone : {metadata.get('phonenumber', 'Not provided')}")

    # Read token_map from Redis
    stored_tokenmap = get_tokenmap_from_redis(session_id)

    print("\nToken Map (stored in Redis encrypted):")
    for token, value in stored_tokenmap.items():
        print(f"  {token} -> {value}")

    # Read history from Redis
    history = get_history_from_redis(session_id)

    print("\nConversation History (what LLM saw):")
    for i, turn in enumerate(history):
        print(f"  Turn {i + 1}: {turn['content']}")


dashboard_view(session_id)


BANK STAFF DASHBOARD

Customer Contact Details:
  Name  : Not provided
  Email : Not provided
  Phone : Not provided

Token Map (stored in Redis encrypted):

Conversation History (what LLM saw):


## Contact Flow with Redis

In [ ]:
token_map.clear()
entity_counters.clear()
conversation_history = []
session_id = "session_bank_001"


def process_turn_with_redis(user_message):
    global conversation_history

    conversation_history = get_history_from_redis(session_id)

    token_map.clear()
    entity_counters.clear()
    loaded_token_map = get_tokenmap_from_redis(session_id)
    token_map.update(loaded_token_map)

    for token in token_map:
        parts = token.strip("<>").split("_")
        if len(parts) == 2:
            entity_type = parts[0]
            try:
                count = int(parts[1])
                if (
                    entity_type not in entity_counters
                    or count > entity_counters[entity_type]
                ):
                    entity_counters[entity_type] = count
            except ValueError:
                pass

    print("\n" + "=" * 55)
    print(f"User: {user_message}")

    # hard block check
    guardrail = input_guardrail(user_message)
    if guardrail["blocked"]:
        print(f"System: {guardrail['message']}")
        return

    _rag_query, llm_query = create_dual_queries(user_message)

    print(f"LLM sees  : {llm_query}")
    print(f"Token Map : {token_map}")

    conversation_history.append({"role": "user", "content": llm_query})

    metadata = build_metadata_from_tokenmap(token_map)

    print("\nMetadata built from token_map:")
    print(f"  fullname    : {metadata['fullname']}")
    print(f"  email       : {metadata['email']}")
    print(f"  phonenumber : {metadata['phonenumber']}")

    print("\nSaving to Redis (before LLM call)...")
    save_to_redis(session_id, conversation_history, token_map, metadata)

    tokenized_agent_response_for_history, user_facing_agent_response = run_agent()

    conversation_history.append(
        {"role": "assistant", "content": tokenized_agent_response_for_history}
    )

    metadata = build_metadata_from_tokenmap(token_map)
    print("\nSaving to Redis (after LLM call)...")
    save_to_redis(session_id, conversation_history, token_map, metadata)

    print(f"\nFINAL USER-FACING RESPONSE: {user_facing_agent_response}")


# Simulate contact flow conversation

token_map.clear()
entity_counters.clear()
conversation_history = []
session_id = "session_bank_001"
r.delete(f"{session_id}:history")
r.delete(f"{session_id}:token_map")
r.delete(f"{session_id}:metadata")

process_turn_with_redis("Hi I want to talk to someone from the bank")
process_turn_with_redis("My name is Ram Sharma. I'm interested in car loans.")
process_turn_with_redis("My email is ram@gmail.com and my phone is 9841234567.")
process_turn_with_redis("Can you book a meeting for me next week?")
process_turn_with_redis("My credit card is 1234-5678-9012-3456.")

print("\n--- Contact Flow Simulation Complete ---")
dashboard_view(session_id)


User: Hi I want to talk to someone from the bank
LLM sees  : Hi I want to talk to someone from the bank
Token Map : {}

Metadata built from token_map:
  fullname    : None
  email       : None
  phonenumber : None

Saving to Redis (before LLM call)...
  History saved    -> session_bank_001:history
  Content          -> tokens only, no real PII
  Token map saved  -> session_bank_001:token_map (encrypted)
  Content          -> real values, encrypted
  Metadata saved   -> session_bank_001:metadata (encrypted)
  Content          -> structured contact info for dashboard

  LLM prompt includes (user part): 'Hi I want to talk to someone from the bank'

  LLM raw response : I can help you book a meeting. Could you please provide me with your full name, email address, and phone number?
  Output clean, no PII found
  LLM response (sanitized by guardrail): I can help you book a meeting. Could you please provide me with your full name, email address, and phone number?
User sees (from LLM)    : I 

### Verifying PII Protection

In [ ]:
token_map.clear()
entity_counters.clear()
conversation_history.clear()
session_id = "session_bank_002"

# Simulate a new conversation flow with sensitive data
process_turn_with_redis(
    "Hello, my name is Jane Doe and my email is jane.doe@example.com."
)
process_turn_with_redis("I am interested in a loan. My phone number is 555-123-4567.")

# Display the dashboard view for this new session
dashboard_view(session_id)


User: Hello, my name is Jane Doe and my email is jane.doe@example.com.
LLM sees  : Hello, my name is <PERSON_1> and my email is <EMAIL_ADDRESS_1>.
Token Map : {'<EMAIL_ADDRESS_1>': 'jane.doe@example.com', '<PERSON_1>': 'Jane Doe'}

Metadata built from token_map:
  fullname    : Jane Doe
  email       : jane.doe@example.com
  phonenumber : None

Saving to Redis (before LLM call)...
  History saved    -> session_bank_002:history
  Content          -> tokens only, no real PII
  Token map saved  -> session_bank_002:token_map (encrypted)
  Content          -> real values, encrypted
  Metadata saved   -> session_bank_002:metadata (encrypted)
  Content          -> structured contact info for dashboard

  LLM prompt includes (user part): 'Hello, my name is <PERSON_1> and my email is <EMAIL_ADDRESS_1>.'

  LLM raw response : Thank you for providing your name and email! Could you please share your phone number so I can assist you further?
  Output clean, no PII found
  LLM response (sanitized b

## Full Conversation Simulation

In [ ]:
token_map.clear()
entity_counters.clear()
conversation_history.clear()
session_id = "session_bank_003"

print("\n--- Starting new conversation simulation ---")

process_turn_with_redis("Hi there! I'm interested in opening a new account.")
process_turn_with_redis("My name is John Smith.")
process_turn_with_redis("You can reach me at john.smith@example.com.")
process_turn_with_redis(
    "My phone number is 123-456-7890. Can you tell me about savings account rates?"
)

print("\n--- Conversation simulation complete ---")

dashboard_view(session_id)


--- Starting new conversation simulation ---

User: Hi there! I'm interested in opening a new account.
LLM sees  : Hi there! I'm interested in opening a new account.
Token Map : {}

Metadata built from token_map:
  fullname    : None
  email       : None
  phonenumber : None

Saving to Redis (before LLM call)...
  History saved    -> session_bank_003:history
  Content          -> tokens only, no real PII
  Token map saved  -> session_bank_003:token_map (encrypted)
  Content          -> real values, encrypted
  Metadata saved   -> session_bank_003:metadata (encrypted)
  Content          -> structured contact info for dashboard

  LLM prompt includes (user part): 'Hi there! I'm interested in opening a new account.'

  LLM raw response : I can assist you with that! Could you please provide me with your full name, phone number, and your email address? Also, if you have a specific type of account in mind (e.g., savings, current), please let me know.
  Output clean, no PII found
  LLM respo

## PII Protection Flow Explained

This section details the step-by-step process of how user input is handled, PII is protected, and how the LLM interacts with the system without ever seeing raw sensitive data.

Here's the flow:

1.  **User Sends a Message**:
    *   The user types their message, which may contain PII.
    *   Example: `"Hi, my name is John Smith. My email is john.smith@example.com."`

2.  **Hard Block Guardrail** (`input_guardrail` function):
    *   The raw user message first goes through `input_guardrail`.
    *   This function uses `analyzer.analyze` to detect predefined hard-block entities (like `CREDIT_CARD`).
    *   **Action**: If a hard-block entity is found, the system immediately blocks the message, returns an error to the user, and the process stops. The LLM never sees this message.
    *   **Output**: `{'blocked': False}` (if no hard-block PII) or `{'blocked': True, 'reason': 'CREDIT_CARD', 'message': '...'}`.

3.  **Tokenization and Dual Query Creation** (`tokenize_text` and `create_dual_queries` functions):
    *   If not blocked, the user's message is passed to `create_dual_queries`.
    *   Inside `create_dual_queries`, the `tokenize_text` function is called:
        *   It uses `analyzer.analyze` to identify all specified PII entities (e.g., `PERSON`, `EMAIL_ADDRESS`, `PHONE_NUMBER`).
        *   Each detected PII is replaced with a unique token (e.g., `"<PERSON_1>"`, `"<EMAIL_ADDRESS_1>"`).
        *   A `token_map` dictionary is maintained, storing the mapping of `token -> real_value` (e.g., `"<PERSON_1>" -> "John Smith"`).
        *   This tokenized version becomes the `llm_query` (what the LLM will see).
    *   `create_dual_queries` also generates a `rag_query` (if needed, typically by selectively restoring some tokens for retrieval-augmented generation).
    *   **Output**: `llm_query` (tokenized message) and `rag_query`.

4.  **Update Conversation History**:
    *   The `llm_query` (the tokenized message) is added to the `conversation_history` list.
    *   **Crucially**: The `conversation_history` *never* stores the raw, original user message or any real PII.

5.  **Build Metadata** (`build_metadata_from_tokenmap` function):
    *   After tokenization, the `build_metadata_from_tokenmap` function is used to create structured PII (e.g., a dictionary with `"fullname"`, `"email"`, `"phonenumber"`) from the `token_map`.
    *   **Purpose**: This structured metadata is useful for internal bank staff dashboards or other systems that need to access PII in an organized manner.

6.  **Secure Storage to Redis** (`save_to_redis` function):
    *   The `save_to_redis` function is called to persist the conversation state:
        *   The `conversation_history` (containing *only* tokens) is saved to Redis.
        *   The `token_map` (containing real PII) is **encrypted** using `Fernet` and then saved to Redis.
        *   The `metadata` (structured PII) is also **encrypted** and saved to Redis.
    *   **Security**: All sensitive data is either tokenized or encrypted before being stored, preventing exposure even if Redis is compromised.

7.  **LLM Interaction** (`run_agent` function):
    *   The `run_agent` function prepares the messages for the LLM, including the tokenized `llm_query`.
    *   The LLM (`client.chat.completions.create`) receives and processes *only* the tokenized `conversation_history` and `llm_query`.
    *   **LLM's Perspective**: From the LLM's point of view, it is interacting with a user who has a name like `<PERSON_1>` and an email like `<EMAIL_ADDRESS_1>`. It is completely unaware of the real values.

8.  **Tool Call Interception and Execution** (`restore_tokens` within `run_agent`):
    *   If the LLM decides to call a tool (e.g., `book_meeting`), it provides parameters using the tokens (e.g., `{"name": "<PERSON_1>", "email": "<EMAIL_ADDRESS_1>"}`).
    *   The system intercepts this tool call:
        *   It uses the `restore_tokens` function to look up the real values from the `token_map` and replace the tokens in the tool parameters.
        *   The tool is then executed with these restored *real* values (e.g., `book_meeting(name="John Smith", email="john.smith@example.com")`).
    *   **Key Point**: The LLM *never* sees the real values for tool execution; the system handles the restoration behind the scenes.

9.  **Output Guardrail** (`output_guardrail` function):
    *   After the tool returns a result, this output is passed through the `output_guardrail` function.
    *   It uses `analyzer.analyze` to check if the tool accidentally generated or leaked any PII in its response.
    *   **Action**: If PII is detected, the `output_guardrail` tokenizes the leaked PII in the LLM's response to prevent it from reaching the user.

10. **Restore for User Display** (`restore_for_user` function):
    *   Finally, the system uses the `restore_for_user` function to replace any tokens in the LLM's response or tool output back to their real values.
    *   **Action**: This ensures the end-user sees a natural, non-tokenized conversation (e.g., "Meeting booked for John Smith.").

11. **Dashboard View** (`dashboard_view` function):
    *   For bank staff, the `dashboard_view` function retrieves the encrypted `metadata` and `token_map` from Redis.
    *   It decrypts this information to display the real customer contact details and the full token mapping.
    *   It also retrieves the `conversation_history` (tokenized) to show what the LLM saw.
    *   **Verification**: This view provides clear proof that the LLM only ever interacts with tokenized data, while real PII is securely stored and only accessible through authorized decryption. This is the ultimate verification that the LLM has never seen the sensitive information.

## Live Chat Simulation


In [ ]:
def reset_session(session_id):
    token_map.clear()
    entity_counters.clear()
    conversation_history.clear()
    # Clear session specific data from Redis
    r.delete(f"{session_id}:history")
    r.delete(f"{session_id}:token_map")
    r.delete(f"{session_id}:metadata")
    print(f"Session {session_id} reset and Redis data cleared.")


def simulate_chat_turn(user_message, session_id):
    global conversation_history

    # Load session state at the beginning of the turn
    conversation_history = get_history_from_redis(session_id)
    token_map.clear()
    entity_counters.clear()
    loaded_token_map = get_tokenmap_from_redis(session_id)
    token_map.update(loaded_token_map)

    # Reconstruct entity_counters based on loaded_token_map
    for token in token_map:
        parts = token.strip("<>").split("_")
        if len(parts) == 2:
            entity_type = parts[0]
            try:
                count = int(parts[1])
                if (
                    entity_type not in entity_counters
                    or count > entity_counters[entity_type]
                ):
                    entity_counters[entity_type] = count
            except ValueError:
                pass

    # Step 1 - hard block check
    guardrail = input_guardrail(user_message)
    if guardrail["blocked"]:
        return guardrail["message"]

    # Step 2 - create dual queries (will update global token_map and entity_counters)
    _rag_query, llm_query = create_dual_queries(user_message)

    # Step 3 - save llm_query to history
    conversation_history.append({"role": "user", "content": llm_query})

    # Step 4: build metadata from token_map
    metadata = build_metadata_from_tokenmap(token_map)

    # Step 5: save everything to Redis (history, token_map, metadata)
    save_to_redis(session_id, conversation_history, token_map, metadata)

    # Step 6: Run agent to get LLM response
    tokenized_agent_response_for_history, user_facing_agent_response = run_agent()

    # Step 7: Append agent's tokenized response to history
    conversation_history.append(
        {"role": "assistant", "content": tokenized_agent_response_for_history}
    )

    # Step 8: Save everything again (token_map might have been updated by output_guardrail in run_agent)
    metadata = build_metadata_from_tokenmap(token_map)
    save_to_redis(session_id, conversation_history, token_map, metadata)

    return user_facing_agent_response


session_id = "live_chat_session_001"
reset_session(session_id)

user_message_1 = "Hi, my name is Alice Wonderland. I'm interested in car loans."
response_1 = simulate_chat_turn(user_message_1, session_id)
print(f"\nFINAL USER-FACING RESPONSE: {response_1}")

dashboard_view(session_id)

user_message_2 = (
    "You can reach me at alice@example.com, and my phone number is 9856789911."
)
response_2 = simulate_chat_turn(user_message_2, session_id)
print(f"\nFINAL USER-FACING RESPONSE: {response_2}")

dashboard_view(session_id)

user_message_3 = "Can you please book a meeting for me next week?"
response_3 = simulate_chat_turn(user_message_3, session_id)
print(f"\nFINAL USER-FACING RESPONSE: {response_3}")

dashboard_view(session_id)

user_message_4 = "My credit card is 1234-5678-9012-3456."
response_4 = simulate_chat_turn(user_message_4, session_id)
print(f"\nFINAL USER-FACING RESPONSE: {response_4}")

dashboard_view(session_id)

user_message_5 = "What are the current FD rates?"
response_5 = simulate_chat_turn(user_message_5, session_id)
print(f"\nFINAL USER-FACING RESPONSE: {response_5}")

dashboard_view(session_id)

print("\n--- Live Chat Simulation Complete ---")

Session live_chat_session_001 reset and Redis data cleared.
  History saved    -> live_chat_session_001:history
  Content          -> tokens only, no real PII
  Token map saved  -> live_chat_session_001:token_map (encrypted)
  Content          -> real values, encrypted
  Metadata saved   -> live_chat_session_001:metadata (encrypted)
  Content          -> structured contact info for dashboard

  LLM prompt includes (user part): 'Hi, my name is <PERSON_1>. I'm interested in car loans.'

  LLM raw response : I can help you create a lead for the car loan interest. Could you please provide me with your phone number?
  Output clean, no PII found
  LLM response (sanitized by guardrail): I can help you create a lead for the car loan interest. Could you please provide me with your phone number?
User sees (from LLM)    : I can help you create a lead for the car loan interest. Could you please provide me with your phone number?
  History saved    -> live_chat_session_001:history
  Content        

In [ ]:
# Interactive Chat Simulation
print("\n--- Starting Interactive Chat Simulation ---")
print("Type 'exit' to end the chat and view the dashboard.\n")

interactive_session_id = "interactive_chat_session_001"

# Reset the session for a clean start
reset_session(interactive_session_id)

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    simulate_chat_turn(user_input, interactive_session_id)

print("\n--- Interactive Chat Simulation Complete ---")

# Display the dashboard view
dashboard_view(interactive_session_id)


--- Starting Interactive Chat Simulation ---
Type 'exit' to end the chat and view the dashboard.

Session interactive_chat_session_001 reset and Redis data cleared.
You: hello
  History saved    -> interactive_chat_session_001:history
  Content          -> tokens only, no real PII
  Token map saved  -> interactive_chat_session_001:token_map (encrypted)
  Content          -> real values, encrypted
  Metadata saved   -> interactive_chat_session_001:metadata (encrypted)
  Content          -> structured contact info for dashboard

  LLM prompt includes (user part): 'hello'

  LLM raw response : Hello! How can I assist you today?
  Output clean, no PII found
  LLM response (sanitized by guardrail): Hello! How can I assist you today?
User sees (from LLM)    : Hello! How can I assist you today?
  History saved    -> interactive_chat_session_001:history
  Content          -> tokens only, no real PII
  Token map saved  -> interactive_chat_session_001:token_map (encrypted)
  Content          ->